# 1226. The Dining Philosophers

- Concept: Deadlock avoidance.
- ROI: Very high. Classic deadlock and starvation control with direct transfer to lock-ordering and resource allocation design.
- Focus: deadlock prevention, fairness, fork acquisition discipline, and liveness arguments instead of just local correctness.
- AI systems mapping: GPU slot coordination, multi-resource locking, and tool/runtime components that need more than one shared resource.


In [ ]:
def test(solution_cls):
    log = []
    d = solution_cls()
    d.wantsToEat(
        0,
        lambda: log.append('pickL'),
        lambda: log.append('pickR'),
        lambda: log.append('eat'),
        lambda: log.append('putL'),
        lambda: log.append('putR'),
    )
    assert 'eat' in log
    assert log.count('pickL') == 1 and log.count('pickR') == 1


In [ ]:
def current_solution():
    return DiningPhilosophers()

result = 'PASS (No solution provided to execute)'
print(result)
# When DiningPhilosophers is runnable with a threaded harness, replace the two lines above with:
# test(current_solution)
# print('PASS')





In [ ]:

from threading import Lock
from concurrent.futures import ThreadPoolExecutor
from abc import ABC, abstractmethod
from enum import auto
from typing import Dict




class AbstractTableState(ABC):
    # owns the runtime resolution of the valid eating states
    @abstractmethod
    def __init__(self, num_philosophers):
        pass

    @abstractmethod
    def setup_table(self):
        pass


class ForkState(Enum):
    FREE = auto()
    USED = auto()



class TableState(AbstractTableState): 
    # owns the runtime resolution of the valid eating states
    def __init__(self, num_philosophers):
        self.num_philosophers = 5 
        self.forks = self.setup_table()
        # assume the forks and plates are derived variables by the problem setup
    def setup_table(self) -> dict: 
        #create a ring of forks alternating with plates.
        forks = {i: ForkState.FREE for i in range(self.num_philosophers)}
        return forks
    
    def get_table_state(self):
        return self.forks

    def set_fork_state(self, fork: int, fork_state: ForkState):
        self.forks[fork] = fork_state
    
    def get_fork_state(self, fork: int) -> ForkState:
        return self.forks[fork]
# ownership of the table run time and valid eating.

class AbstractTableManager(ABC):
    def __init__(self, num_philosophers = 5):
        self.table = TableState(num_philosophers)
        self.lock = Lock()

class TableManager: 
    def __init__(self, num_philosophers = 5):
        super().__init__(num_philosophers)

    def eat(self, philosopher: int) -> Dict[str, bool]:
        with self.lock:
            left_fork = (philosopher - 1) % self.num_philosophers
            right_fork = philosopher % self.num_philosophers
            if self.table.get_fork_state(left_fork) == ForkState.FREE and self.table.get_fork_state(right_fork) == ForkState.FREE:
                self.table.set_fork_state(left_fork, ForkState.USED)
                self.table.set_fork_state(right_fork, ForkState.USED)
                return {"Result": True}
 
        
            
                
                






class DinningPhilosopherSimulator:
    def __init__(self, num_philosophers = 5):
        self.num_philosophers = num_philosophers
        

    def runtime(self):
        with ThreadPoolExecutor(max_workers=self.num_philosophers) as executor:
            futures = [executor.submit(self.eat) for _ in range(self.num_philosophers)]

    def eat(self):
        # eat logic here
        pass



## I realized that Leetcode exercise already has the abstractions made for me.

- I'm thinking that the leetcode abstraction boundary design is good for differing algorithms where I choose how to put together the abstractions

In [ ]:

from threading import Lock, Event
from concurrent.futures import ThreadPoolExecutor
from abc import ABC, abstractmethod
from enum import auto
from typing import Dict

class AbstractTableState(ABC):
    # owns the runtime resolution of the valid eating states
    @abstractmethod
    def __init__(self, num_philosophers):
        pass

    @abstractmethod
    def setup_table(self):
        pass

# can be replaced with event
# class ForkState(Enum):
#     FREE = auto()
#     USED = auto()

# we have a circular 
class TableState(AbstractTableState): 
    # owns the runtime resolution of the valid eating states
    def __init__(self, num_philosophers):
        self.num_philosophers = 5 
        self.free_forks = self.setup_table()
        # assume the forks and plates are derived variables by the problem setup
    def setup_table(self) -> dict: 
        #create a ring of forks alternating with plates.
        forks = {i: Event() for i in range(self.num_philosophers)}
        for event in forks.values():
            event.set()
        return forks
    
    def get_table_state(self):
        return self.free_forks

    def set_fork_free(self, fork: int):
        self.free_forks[fork].set()
        
    def set_fork_used(self, fork:int):
        self.free_forks[fork].clear()
    
    def wait_fork(self, fork: int) -> None:
        self.free_forks[fork].wait()

    def get_num_philosophers(self):
        return self.num_philosophers
    






class DiningPhilosophers:
    def __init__(self, num_philosophers=5):
        self.table_state = TableState(num_philosophers)
    
    def _resolve_philosopher_forks(self, philosopher: int):
        left_fork = philosopher
        right_fork = (philosopher + 1) % self.table_state.get_num_philosophers()
        return left_fork, right_fork
        
    def wantsToEat(self,
                   philosopher: int,
                   pickLeftFork: 'Callable[[], None]',
                   pickRightFork: 'Callable[[], None]',
                   eat: 'Callable[[], None]',
                   putLeftFork: 'Callable[[], None]',
                   putRightFork: 'Callable[[], None]') -> None:

        left_fork, right_fork = self._resolve_philosopher_forks()
        self.table_state.wait_fork(left_fork)
        
        

